# Looped Qwen3-1.7B: build, distill, evaluate

Turns Qwen3-1.7B's 28 layers into a looped student (unique first and last layers, one shared middle block run L times with a LoRA per loop) and distills it from the frozen original on FineWeb-Edu. `README.md` next to this notebook explains every stage.

This notebook is a thin driver for `run.py`, the same entrypoint `colab exec` runs headless.

1. Upload this repo's files to `MyDrive/looped-distill/code/`.
2. Runtime → Change runtime type → **A100 GPU**.
3. Set `RUN_ARGS` below (`["--smoke"]` for the first run; `python run.py --help` lists every option).
4. Runtime → **Run all**, and approve the Google sign-in and the Drive mount.

In [ ]:
CODE_DIR = "/content/drive/MyDrive/looped-distill/code"
RUN_ARGS = ["--smoke"]
RELEASE_GPU_AT_END = False

## Sign in and mount Drive

`authenticate_user` leaves Google credentials on the VM; every stage uses them to write the run folder through the Drive API. The Drive mount is only used to read the code.

In [ ]:
import sys

from google.colab import auth, drive

auth.authenticate_user()
drive.mount("/content/drive")
sys.path.insert(0, CODE_DIR)
from run import run_command

## Install pinned packages

The versions in `requirements.txt` are the ones the scaffold was written and checked against. Every stage runs as its own process, so the notebook never imports the old torch.

In [ ]:
run_command(sys.executable, "-m", "pip", "install", "-q", "uv==0.13.7")
run_command("uv", "pip", "install", "--python", sys.executable, "-q", "-r", f"{CODE_DIR}/requirements.txt")

## Run

`run.py` prints the GPU (and refuses one under 39 GB for a full run), checks the Drive credentials, then runs `model.py`, `distill.py`, `eval.py` and the final `storage.py` verify, and prints `RUN COMPLETE` once every file on Drive matches `manifest.json`.

In [ ]:
run_command(sys.executable, "-u", f"{CODE_DIR}/run.py", *RUN_ARGS)

In [ ]:
if RELEASE_GPU_AT_END:
    from google.colab import runtime

    runtime.unassign()